# Domain Randomization Individual Ablation

This notebook has the initial work for the ablation stufy to evaluate each individual Domain Randomization technique. 

In [1]:
from deepracer_genesis.randomization.catalog import CATALOG, BY_NAME, by_layer
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Domain randomization orgnized by type

In [2]:
drand_types = (
    "image",
    "physics",
    "geometry",
    "visual",
    "actuation"
)

list_names = lambda knobs: [knob.name for knob in knobs]

drands = {
    name: pd.Series(list_names(by_layer(name)))
    for name in drand_types
}

pd.DataFrame(drands).fillna("-")

,image,physics,geometry,visual,actuation
0,brightness,friction,track_width_scale,world_color,steer_noise
1,contrast,mass_shift,-,camera_pitch_jitter,speed_noise
2,saturation,com_shift,-,camera_pos_jitter,delay_steps
3,hue,steer_kp_scale,-,pixel_noise,-
4,blur,wheel_kv_scale,-,env_map_tint,-
5,cutout,armature,-,env_map_multiplier,-
6,noise,-,-,-,-
7,gamma,-,-,-,-
8,white_balance,-,-,-,-
9,vignette,-,-,-,-


In [3]:
from deepracer_genesis.experiment import (
    AsymmetricCameraPolicy,
    CameraEnvironment,
    FeatureEnvironment,
    DomainRandomizationActions,
    DomainRandomizationCamera,
    DomainRandomizationPhysics,
    DomainRandomizationTrackAppearance,
    Evaluation,
    PPO
)

from deepracer_genesis.randomization.spaces import FloatRange, IntRange
from deepracer_genesis.tools.zoo import view_zoo

import optuna

In [4]:
SEED = 0
ROOT = "runs/best_camera"
WALL_BUDGET_H = 10.0
REUSE = False                      # return recorded stage results from cache

# renderer bench
BENCH_STEPS = 150
NYX_SLOWDOWN_OK = 2.0              # prefer nyx unless > this x slower

# HPO (no-DR, single track: measures pure learning ability)
HPO_TRIALS = 16
HPO_DEADLINE_H = 3.4
TRIAL_STEPS = 1_200_000
TRIAL_EVAL_EVERY = 600_000
TRIAL_TRACK = "reinvent_base"

# final training (winner config + zoo + full DR)
FINAL_STEPS_MAX = 30_000_000
FINAL_STEPS_MIN = 2_000_000
FINAL_RESERVE_H = 1.0
FINAL_EVALS = 6

RESOLUTION = (160, 120)            # physical-camera parity

In [1]:
SEARCH_SPACE = {
    "lr": FloatRange(1e-4, 1e-2, log=True),
    "entropy_coef": FloatRange(1e-3, 3e-2, log=True),
    "epochs": IntRange(3, 8),
    "clip": FloatRange(0.1, 0.3),
    "activation": "relu",
}

def suggest_architecture(tiral: optuna.Trial) -> dict:
    """
    Search space for the architecture

    trial: Optuna Trial.
    """
    depth = trial.suggest_int("depth",2,6)
    width = trial.suggest_categorical("width", [128,256,521])
    return tuple(max(width // 2**i, 32) for i in range(depth))

NameError: name 'FloatRange' is not defined